# Matrix Multiplications and AM-GM Inequalities

In [ ]:
#@title Verification code

import itertools
import math
import numpy as np
import numba

njit = numba.njit


def get_score(b_matrices: np.ndarray, n: int, m: int, d: int) -> float:
  """Calculates the score for a given set of matrices to test the AM-GM conjecture.

  The score is defined as RHS - LHS of the inequality. A positive score
  indicates a counterexample has been found.
  """
  if not isinstance(b_matrices, np.ndarray) or b_matrices.shape != (n, d, d):
    return -1_000_000.0

  b_matrices = np.clip(b_matrices, -1000.0, 1000.0)
  if np.isnan(b_matrices).any() or np.isinf(b_matrices).any():
    return -1_000_000.0

  try:
    matrices_A = [b @ b.conj().T for b in b_matrices]
  except ValueError:
    return -1_000_000.0

  # Calculate the Left-Hand Side (LHS) of the inequality
  sum_all_norms = 0.0
  for indices in itertools.product(range(n), repeat=m):
    try:
      prod = np.identity(d, dtype=np.complex128)
      for idx in indices:
        prod = prod @ matrices_A[idx]
      norm = np.linalg.norm(prod, ord='nuc')
      if np.isinf(norm) or np.isnan(norm):
        return -1_000_000.0
      sum_all_norms += norm
    except np.linalg.LinAlgError:
      return -1_000_000.0

  lhs = (1.0 / (n**m)) * sum_all_norms

  # Calculate the Right-Hand Side (RHS) of the inequality
  sum_distinct_norms = 0.0
  if n < m:
    return -lhs

  for indices in itertools.permutations(range(n), m):
    try:
      prod = np.identity(d, dtype=np.float64)
      for idx in indices:
        prod = prod @ matrices_A[idx]
      norm = np.linalg.norm(prod, ord='nuc')
      if np.isinf(norm) or np.isnan(norm):
        return -1_000_000.0
      sum_distinct_norms += norm
    except np.linalg.LinAlgError:
      return -1_000_000.0

  num_permutations = math.factorial(n) / math.factorial(n - m)
  rhs = (1.0 / num_permutations) * sum_distinct_norms

  score = rhs - lhs
  if np.isnan(score) or np.isinf(score):
    return -1_000_000.0

  return score

In [ ]:
#@title Initial program

import time
import numpy as np


def search_for_best_matrices(n: int, m: int, d: int):
  """Searches for a set of matrices that constitute a counterexample.

  The object being searched for is a numpy array of shape (n, d, d),
  representing n matrices of size d x d.
  """
  b_matrices = np.random.randn(n, d, d) + 1j * np.random.randn(n, d, d)

  best_score = get_score(b_matrices, n, m, d)
  best_b_matrices = b_matrices.copy()

  start_time = time.time()
  while time.time() - start_time < 200:
    index_to_mutate = np.random.randint(0, n)
    complex_noise = (np.random.randn(d, d) + 1j * np.random.randn(d, d)) * 0.5
    b_matrices[index_to_mutate] += complex_noise
    score = get_score(b_matrices, n, m, d)

    if score > best_score:
      best_score = score
      best_b_matrices = b_matrices.copy()

    if np.random.rand() < 0.2:
      b_matrices = best_b_matrices.copy()

  return best_b_matrices

**Prompt used**

Disproving the Non-commutative AM-GM Conjecture

Act as an expert in matrix analysis and optimization. Your task is to find a counterexample to the following conjecture:

For positive-semidefinite matrices $A_1, \ldots, A_n$ and any unitarily invariant norm $|||\cdot|||$ (including the operator norm and Schatten $p$-norms), the following holds for each integer $m \leq n$:
[
1/n^m \sum_(j_1, j_2, \ldots, j_m = 1)^n |||A_(j_1)A_(j_2)\ldots A_(j_m)||| \geq (n-m)!/n! \sum_(j_1, j_2, \ldots, j_m = 1 (all distinct))^n |||A_(j_1)A_(j_2)\ldots A_(j_m)|||
]

The case $m>=4$ is open, and your goal is to find a counterexample for it. You will be given parameters n (the number of matrices), m (the length of the products), and d (the dimension of the matrices).

Your task is to write a search function that finds a set of n square matrices of dimension d, let's call them $B_1, \ldots, B_n$. These matrices will be used to generate the positive-semidefinite matrices as $A_i = B_i B_i^T$. Your search should aim to find a set of $B_i$ matrices that violates the inequality. The entire collection of n matrices of size d x d will be represented as a single NumPy array of shape (n, d, d).

Your construction will be evaluated by the following score function, which calculates RHS - LHS of the inequality. A positive score means you have successfully found a counterexample.

def get_score(b_matrices: np.ndarray, n: int, m: int, d: int) -> float:
    """
    Calculates the score for a given set of matrices to test the AM-GM conjecture.
    (The implementation is provided in the environment.)
    """
    # ... implementation is provided ...

You may code up any search method you want, and you are allowed to call the get_score() function as many times as you want. You have access to it; you don't need to code it up. You want the score it gives you to be as high (positive) as possible!

Your task is to write a search_for_best_matrices function that takes n, m, and d as input and searches for the best NumPy array of shape (n, d, d). Your function has 200 seconds to run. If it hasn't returned anything after that time, it will be terminated with a score of negative infinity. Structure your search within a time-limited loop, e.g., while time.time() - start_time < 200:.

Good luck, I believe in you!

## What AlphaEvolve found

AlphaEvolve was used to search for counterexamples to Duchi's conjecture that $C(n,m,d) = 1$ for all $n, m, d$, focusing on various parameter choices with $m \geq 4$ (the open case) and several unitarily invariant norms (Schatten $k$-norms for $k \in \{1,2,3,\infty\}$ and Ky Fan 2- and 3-norms). AlphaEvolve was able to find constructions attaining the upper bound $C(n,m,d) \leq 1$ but was not able to find any constructions improving this bound, i.e., it did not find a counterexample to the conjecture.